# RAG (Retrieval-Augmented Generation)

Modelos de linguagem possuem conhecimento vasto, mas não sabem nada sobre os dados privados da sua empresa: documentos internos, políticas, manuais, bases de conhecimento. Se você perguntar ao modelo sobre as taxas do seu produto, ele não vai saber responder.

**RAG** resolve esse problema em três passos:
1. **Recuperar** os trechos mais relevantes dos seus documentos
2. **Aumentar** o prompt com esses trechos como contexto
3. **Gerar** a resposta usando apenas as informações fornecidas

O modelo não precisa ser treinado novamente. Ele simplesmente recebe a informação certa na hora certa.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Carregando documentos

O primeiro passo é carregar os documentos que formarão a base de conhecimento. O LangChain oferece diversos loaders para diferentes formatos (PDF, Markdown, HTML, CSV, etc.).

Vamos carregar um arquivo Markdown com informações da **NovaTech Financeira**, uma fintech fictícia.

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("resources/base_conhecimento.md")
documentos = loader.load()

print(f"Documentos carregados: {len(documentos)}")
print(f"Tamanho: {len(documentos[0].page_content)} caracteres")

Documentos carregados: 1
Tamanho: 5810 caracteres


In [3]:
print(documentos[0].page_content[:500])

# NovaTech Financeira - Base de Conhecimento Interna

## Sobre a Empresa

A NovaTech Financeira é uma fintech brasileira fundada em 2020, com sede em São Paulo. A empresa oferece soluções financeiras digitais para pessoas físicas e pequenas empresas. Atualmente conta com mais de 2 milhões de clientes ativos e opera exclusivamente de forma digital, sem agências físicas.

A missão da NovaTech é democratizar o acesso a serviços financeiros por meio de tecnologia, oferecendo produtos simples, transp


O documento inteiro foi carregado como um único bloco de texto com quase 6 mil caracteres. Mas modelos de linguagem têm limite de contexto, e passar o documento inteiro seria ineficiente e caro. Precisamos dividir em pedaços menores.

## Dividindo em chunks

O **text splitting** divide o documento em pedaços (chunks) de tamanho controlado. Usamos o `RecursiveCharacterTextSplitter`, que tenta dividir respeitando parágrafos, frases e palavras, nessa ordem de prioridade.

Dois parâmetros importantes:
- `chunk_size`: tamanho máximo de cada chunk (em caracteres)
- `chunk_overlap`: sobreposição entre chunks consecutivos (para não perder contexto nas bordas)

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_documents(documentos)
print(f"Chunks criados: {len(chunks)}")

Chunks criados: 7


In [5]:
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content)
    print()

--- Chunk 0 (892 chars) ---
# NovaTech Financeira - Base de Conhecimento Interna

## Sobre a Empresa

A NovaTech Financeira é uma fintech brasileira fundada em 2020, com sede em São Paulo. A empresa oferece soluções financeiras digitais para pessoas físicas e pequenas empresas. Atualmente conta com mais de 2 milhões de clientes ativos e opera exclusivamente de forma digital, sem agências físicas.

A missão da NovaTech é democratizar o acesso a serviços financeiros por meio de tecnologia, oferecendo produtos simples, transparentes e com taxas justas.

## Produtos

### Conta Digital

A Conta Digital NovaTech é gratuita e não cobra taxa de manutenção. Inclui:
- Cartão de débito virtual e físico sem anuidade
- Transferências ilimitadas via Pix
- Até 5 TEDs gratuitas por mês (R$ 8,50 por TED adicional)
- Rendimento automático de 100% do CDI sobre o saldo em conta
- Extrato detalhado em tempo real pelo aplicativo

--- Chunk 1 (965 chars) ---
Para abrir uma conta, o cliente precisa ser maior 

Com chunks maiores (1000 caracteres), cada pedaço mantém mais contexto — seções inteiras como "Cartão de Crédito" ou "Investimentos" tendem a ficar em um único chunk, o que melhora a qualidade da busca.

## Criando o vector store

Agora vamos transformar os chunks em embeddings e armazená-los em um vector store. O método `from_documents` faz tudo de uma vez: gera os embeddings e indexa os documentos.

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

vectorstore = InMemoryVectorStore.from_documents(
    chunks,
    OpenAIEmbeddings(model="text-embedding-3-small")
)

print(f"Vector store criado com {len(chunks)} documentos.")

Vector store criado com 7 documentos.


## Testando a busca

Antes de montar o RAG completo, vamos testar se a busca semântica está funcionando. A pergunta do usuário é convertida em embedding e comparada com os chunks armazenados.

In [7]:
resultados = vectorstore.similarity_search("quais sao as taxas do cartao de credito?", k=3)

for i, doc in enumerate(resultados):
    print(f"--- Resultado {i+1} ---")
    print(doc.page_content[:200])
    print()

--- Resultado 1 ---
Para abrir uma conta, o cliente precisa ser maior de 18 anos, possuir CPF válido e realizar a verificação de identidade pelo aplicativo (selfie + documento).

### Cartão de Crédito

O Cartão de Crédit

--- Resultado 2 ---
### Empréstimos

**Empréstimo Pessoal:**
- Taxas a partir de 1,49% ao mês
- Prazo de 3 a 60 meses
- Valores de R$ 500 a R$ 50.000
- Contratação 100% digital, sem papelada
- Dinheiro na conta em até 24

--- Resultado 3 ---
A taxa de juros do rotativo é de 12,99% ao mês para todas as modalidades. O parcelamento da fatura tem taxa de 5,99% ao mês.

### Investimentos

A plataforma de investimentos da NovaTech oferece:

**R



Com chunks maiores, a busca consegue recuperar seções mais completas. Agora podemos usar esses trechos como contexto para o modelo gerar uma resposta.

## Montando o RAG

O fluxo completo: receber a pergunta, buscar os chunks relevantes, montar um prompt com o contexto e pedir ao modelo para responder baseando-se apenas nessas informações.

In [8]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini", temperature=0)

def perguntar(pergunta: str) -> str:
    # 1. Recuperar chunks relevantes
    docs = vectorstore.similarity_search(pergunta, k=5)
    contexto = "\n\n".join([doc.page_content for doc in docs])

    # 2. Montar prompt com contexto
    prompt = (
        "Responda a pergunta abaixo usando APENAS as informacoes do contexto fornecido. "
        "Se a informacao nao estiver no contexto, diga que nao encontrou a informacao.\n\n"
        f"Contexto:\n{contexto}\n\n"
        f"Pergunta: {pergunta}"
    )

    # 3. Gerar resposta
    resposta = model.invoke(prompt)
    return resposta.content

In [9]:
print(perguntar("Quais sao as taxas do cartao de credito?"))

As taxas do cartão de crédito NovaTech são:

- Anuidade:
  - NovaTech Essencial: sem anuidade
  - NovaTech Plus: R$ 19,90/mês (isenta para gastos acima de R$ 3.000/mês)
  - NovaTech Black: R$ 59,90/mês (isenta para gastos acima de R$ 8.000/mês)

- Taxa de juros do rotativo: 12,99% ao mês para todas as modalidades

- Taxa para parcelamento da fatura: 5,99% ao mês


In [10]:
print(perguntar("Como funciona o CDB da NovaTech?"))

O CDB da NovaTech funciona da seguinte forma:

- Há duas modalidades de CDB oferecidas:
  1. CDB NovaTech: rendimento de 110% do CDI, com liquidez diária e aplicação mínima de R$ 1,00.
  2. CDB de longo prazo: rendimento de 120% do CDI, com vencimento em 2 anos e aplicação mínima de R$ 1.000.

Esses investimentos podem ser acompanhados em tempo real pelo aplicativo e não há taxa de custódia.


In [11]:
print(perguntar("Qual o prazo para cancelar uma conta?"))

O prazo para efetivação do cancelamento da conta é de até 5 dias úteis.


## Testando os limites

Algumas perguntas podem não ser respondidas corretamente — seja porque a informação realmente não está nos documentos, ou porque os chunks recuperados não contêm o trecho certo. Isso é esperado: a qualidade do RAG depende diretamente da qualidade do chunking e da busca.

Vamos testar com uma pergunta cuja resposta **não existe** nos documentos.

In [12]:
print(perguntar("Qual e a receita anual da NovaTech?"))

Não encontrou a informação sobre a receita anual da NovaTech no contexto fornecido.


O modelo reconhece que a informação não está no contexto e avisa, em vez de inventar uma resposta. Isso é fundamental para sistemas confiáveis.

Repare que o RAG não é perfeito: a qualidade das respostas depende de quais chunks são recuperados. Se o chunk certo não entrar no top-k, o modelo não terá a informação necessária. Em produção, técnicas como **ajustar o chunk_size**, **aumentar o k** ou usar **re-ranking** melhoram significativamente os resultados.

Com esse notebook, temos um RAG funcional: carregamos documentos, dividimos em chunks, indexamos em um vector store e usamos busca semântica para dar contexto ao modelo. No próximo notebook, vamos transformar esse RAG em uma **tool** que um agente pode usar autonomamente.